## 0 Load model and tokenizer



In [ ]:
from huggingface_hub import snapshot_download
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer

In [ ]:
snapshot_download("baffo32/decapoda-research-llama-7B-hf")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 42 files:   0%|          | 0/42 [00:00<?, ?it/s]

'/root/.cache/huggingface/hub/models--baffo32--decapoda-research-llama-7B-hf/snapshots/aa18b48a1330572a6dd5f5d5619ed19838ca285c'

In [ ]:
from huggingface_hub import snapshot_download
import transformers
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL = "/root/.cache/huggingface/hub/models--baffo32--decapoda-research-llama-7B-hf/snapshots/aa18b48a1330572a6dd5f5d5619ed19838ca285c"
device = "cuda" if torch.cuda.is_available() else "cpu"

model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.float16).to(device)
# decapoda's config has broken special token IDs
model.config.bos_token_id = 1
model.config.eos_token_id = 2
model.config.pad_token_id = 0


tokenizer = AutoTokenizer.from_pretrained(
    MODEL, unk_token="<unk>", bos_token="<s>", eos_token="</s>"
)
tokenizer.add_bos_token = True
if hasattr(tokenizer, "update_post_processor"):
    tokenizer.update_post_processor()

print(tokenizer("")["input_ids"])  # should print [1]


# LLaMA marks spaces with '▁', not GPT-2's 'Ġ'
model.to_str_tokens = lambda t: [t.replace('▁', ' ') for t in tokenizer.tokenize(t)]

model.tokenizer = tokenizer
# In steering experimentation spaces were found to work well, this makes no sense and I hate it.
tokenizer.pad_token_id = int(model.tokenizer.encode(" ")[-1])

[transformers] Model config: pad_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got -1. This may result in unexpected behavior.
[transformers] The following generation flags are not valid and may be ignored: ['pad_token_id']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

[1]


## 1 Load Yelp Sentiment Transfer dataset

In [ ]:
!pwd

/content


In [ ]:
!git clone https://github.com/shentianxiao/language-style-transfer.git

Cloning into 'language-style-transfer'...
remote: Enumerating objects: 242, done.
remote: Total 242 (delta 0), reused 0 (delta 0), pack-reused 242 (from 1)
Receiving objects: 100% (242/242), 31.19 MiB | 19.63 MiB/s, done.
Resolving deltas: 100% (140/140), done.


In [ ]:
import random
path0 = './language-style-transfer/data/yelp/sentiment.dev.0'
path1 = './language-style-transfer/data/yelp/sentiment.dev.1'

with open(path0) as f:
  # help(f)
  dev0_sentences= f.readlines()
  print(len(dev0_sentences))

with open(path1) as f:
  dev1_sentences= f.readlines()
  print(len(dev1_sentences))




25278
38205


In [ ]:
dev0_100 = []
dev1_100 = []

for i in range(0,100):
  tmp1 = dev0_sentences[random.randint(0, len(dev0_sentences))]
  dev0_100.append(tmp1)

for i in range(0,100):
  tmp2 = dev1_sentences[random.randint(0, len(dev1_sentences))]
  dev1_100.append(tmp2)



In [ ]:
dev0_sentences[0]


'ok never going back to this place again .\n'

In [ ]:
dev1_sentences[0]

'staff behind the deli counter were super nice and efficient !\n'

In [ ]:
print(dev0_100[0])
print(dev1_100[0])


it was clearly gross .

this place is pretty good on the weekend nights .



## 2 Compute Steering vectors for each sentence

In [ ]:
# add model hook
# loss func, optimiser
# loss forward, bacward zero grad x 500
# return z_grad as a global variable, append z_grad to an array, idxs represent sentence
# need train, loop 200 times

In [ ]:
## how2 access the module list -> how 2 access a single LlamaDecoderLayer
from functools import partial

# res stream : batch, psn , dmodel  | 1, 30, 4096

def add_steering_vector(inst, inp, out, z_steer, timestep):
  z_steer = z_steer.to(out.dtype)
  # print(f"z steer : {z_steer.shape} , {z_steer.mean()}")
  # print(f"out before steering : {out.shape}, {out.mean()}")
  # out = batch, psn , model
  out[:, timestep] =  out[:, timestep] + z_steer
  # print(f"out after sterring : {out.shape}, {out.mean()}")
  return out

In [ ]:
import torch as t
from torch.nn.init import xavier_normal_
from torch.nn.functional import cross_entropy
from torch.optim import Adam
import einops
from einops import rearrange

def train_loop(model, token_ids, z_steer, N, optimizer):       # ADAM: takes optimizer instead of lr
    cross_ent = []
    for i in range(N):
        logits = model(token_ids).logits
        pred = rearrange(logits[:, :-1], "b s v -> b v s")     # FIX: shift, predictions 0..n-2
        target = token_ids[:, 1:]                              # FIX: shift, the next tokens
        loss = cross_entropy(pred, target)
        loss.backward()

        optimizer.step()                                       # ADAM: replaces the manual update
        optimizer.zero_grad()                                  # ADAM: resets z_steer.grad

        cross_ent.append(loss.item())                          # FIX: store number, not tensor
        if i % 20 == 0:
            print(f"step {i}, loss = {loss.item():.4f}")
        if loss.item() < 0.01:                                 # early stop once learned
            break
    return cross_ent

def steer_model(model, reference, int_layer=6):
    """
    Optimize a steering vector so the frozen model reconstructs `reference`.
    z_steer is added to the output of layer `int_layer` at the first timestep.

    returns:
        cross_ent: list of losses
        z_steer: the trained steering vector
    """
    model = model.to(device)
    model.requires_grad_(False)                                # FIX: freeze the model

    cfg = model.config
    z_steer = t.empty(1, cfg.hidden_size, device=device)
    xavier_normal_(z_steer)
    z_steer.requires_grad_(True)

    inputs = tokenizer(reference, return_tensors="pt")         # FIX: no " ".join on a string
    eos = t.tensor([[tokenizer.eos_token_id]])
    token_ids = t.cat([inputs.input_ids, eos], dim=1).to(device)

    N = 500
    lr = 0.1
    optimizer = Adam([z_steer], lr=lr)                         # ADAM: create optimizer

    for m in model.modules():
        m._forward_hooks.clear()
        m._forward_pre_hooks.clear()

    hook = partial(add_steering_vector, z_steer=z_steer, timestep=0)   # FIX: 6 → 0
    handle = model.model.layers[int_layer].register_forward_hook(hook)

    try:
        cross_ent = train_loop(model, token_ids, z_steer, N, optimizer)
    finally:
        handle.remove()                                        # FIX: always remove the hook

    return cross_ent, z_steer.detach()

prompt = dev0_100[0]
cross_ent, z_steer = steer_model(model, prompt)
                                         # should end near 0

step 0, loss = 5.9375
step 20, loss = 2.4004
step 40, loss = 1.2480
step 60, loss = 0.6401
step 80, loss = 0.3142


KeyboardInterrupt: 

## G1 Steer Batch

In [ ]:
from torch.nn.utils.rnn import pad_sequence

def steer_batch(model, sentences, int_layer=6, N=500, lr=0.1):
    B = len(sentences)
    eos = t.tensor([tokenizer.eos_token_id])
    seqs = [t.cat([tokenizer(s, return_tensors="pt").input_ids[0], eos]) for s in sentences]

    token_ids = pad_sequence(seqs, batch_first=True, padding_value=tokenizer.eos_token_id).to(device)
    labels = pad_sequence(seqs, batch_first=True, padding_value=-100).to(device)   # -100 = ignore
    attn = (labels != -100).long()

    z_steer = t.empty(B, model.config.hidden_size, device=device)
    xavier_normal_(z_steer)
    z_steer.requires_grad_(True)
    optimizer = Adam([z_steer], lr=lr)

    for m in model.modules():
        m._forward_hooks.clear()
    handle = model.model.layers[int_layer].register_forward_hook(
        partial(add_steering_vector, z_steer=z_steer, timestep=0))

    try:
        for i in range(N):
            logits = model(token_ids, attention_mask=attn).logits
            pred = rearrange(logits[:, :-1], "b s v -> b v s")
            target = labels[:, 1:]
            tok_loss = cross_entropy(pred.float(), target, ignore_index=-100, reduction="none")
            row_loss = tok_loss.sum(1) / (target != -100).sum(1)       # mean loss per sentence

            row_loss.sum().backward()          # each row's gradient only affects its own vector
            optimizer.step()
            optimizer.zero_grad()

            if i % 20 == 0:
                print(f"step {i}, worst = {row_loss.max().item():.4f}, mean = {row_loss.mean().item():.4f}")
            if row_loss.max().item() < 0.01:   # stop when every sentence has converged
                break
    finally:
        handle.remove()

    return z_steer.detach().cpu(), row_loss.detach().cpu()

In [ ]:
batch_size = 32
results = {}

for start in range(0, len(dev0_100), batch_size):
    chunk = dev0_100[start:start + batch_size]
    Z, losses = steer_batch(model, chunk)
    for s, z, l in zip(chunk, Z, losses):
        results[s] = {"z": z.unsqueeze(0), "final_loss": l.item()}
    t.save(results, "z_steers.pt")

step 0, worst = 7.9734, mean = 5.5657
step 20, worst = 3.3905, mean = 2.2447
step 40, worst = 1.5629, mean = 0.8766
step 60, worst = 0.8778, mean = 0.3817
step 80, worst = 0.5466, mean = 0.1764
step 100, worst = 0.2748, mean = 0.0852
step 120, worst = 0.1453, mean = 0.0448
step 140, worst = 0.0837, mean = 0.0248
step 160, worst = 0.0573, mean = 0.0148
step 180, worst = 0.0410, mean = 0.0098
step 200, worst = 0.0290, mean = 0.0069
step 220, worst = 0.0199, mean = 0.0051
step 240, worst = 0.0118, mean = 0.0039
step 0, worst = 8.4550, mean = 5.5478
step 20, worst = 3.3506, mean = 2.2184
step 40, worst = 1.9795, mean = 1.0217
step 60, worst = 1.2384, mean = 0.4897
step 80, worst = 0.8020, mean = 0.2242
step 100, worst = 0.6063, mean = 0.0987
step 120, worst = 0.4016, mean = 0.0505
step 140, worst = 0.2047, mean = 0.0278
step 160, worst = 0.1150, mean = 0.0164
step 180, worst = 0.0871, mean = 0.0106
step 200, worst = 0.0672, mean = 0.0077
step 220, worst = 0.0533, mean = 0.0059
step 240, wo

In [ ]:
batch_size = 8
results = {}

for start in range(0, len(dev1_100), batch_size):
    chunk = dev0_100[start:start + batch_size]
    Z, losses = steer_batch(model, chunk)
    for s, z, l in zip(chunk, Z, losses):
        results[s] = {"z": z.unsqueeze(0), "final_loss": l.item()}
    t.save(results, "z_steers_target.pt")

In [ ]:
model.config

LlamaConfig {
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 1,
  "dtype": "float16",
  "eos_token_id": 2,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 4096,
  "initializer_range": 0.02,
  "intermediate_size": 11008,
  "max_position_embeddings": 2048,
  "max_sequence_length": 2048,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 32,
  "num_hidden_layers": 32,
  "num_key_value_heads": 32,
  "pad_token_id": 0,
  "pretraining_tp": 1,
  "rms_norm_eps": 1e-06,
  "rope_parameters": {
    "rope_theta": 10000.0,
    "rope_type": "default"
  },
  "tie_word_embeddings": false,
  "transformers_version": "5.22.0",
  "use_cache": true,
  "vocab_size": 32000
}

In [ ]:
help(transformers.models.llama.tokenization_llama.LlamaTokenizer.encode)

Help on function encode in module transformers.tokenization_utils_base:

encode(
    self,
    text: 'TextInput | PreTokenizedInput | EncodedInput',
    text_pair: 'TextInput | PreTokenizedInput | EncodedInput | None' = None,
    add_special_tokens: 'bool' = True,
    padding: 'bool | str | PaddingStrategy' = False,
    truncation: 'bool | str | TruncationStrategy | None' = None,
    max_length: 'int | None' = None,
    stride: 'int' = 0,
    padding_side: 'str | None' = None,
    return_tensors: 'str | TensorType | None' = None,
    **kwargs
) -> 'list[int]'
    Converts a string to a sequence of ids (integer), using the tokenizer and vocabulary.

    Same as doing `self.convert_tokens_to_ids(self.tokenize(text))`.

    Args:
        text (`str`, `list[str]` or `list[int]`):
            The first sequence to be encoded. This can be a string, a list of strings (tokenized string using the
            `tokenize` method) or a list of integers (tokenized string ids using the `convert_token

In [ ]:
model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 4096, padding_idx=31999)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (v_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=4096, out_features=11008, bias=False)
          (up_proj): Linear(in_features=4096, out_features=11008, bias=False)
          (down_proj): Linear(in_features=11008, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-06)
        (post_attention_layernorm): LlamaRMSNorm((4096,), eps=1e-06)
      )
    )
    (norm): LlamaRMSNorm((409

## L1 Building loop from scratch

In [ ]:
from torch.optim import Adam
from torch.nn.init import xavier_uniform_
from torch.nn.functional import cross_entropy
import einops
from einops import rearrange
from functools import partial
from pprint import pprint

def add_steering_vector(inst, inp, out, z_steer, timestep):
  z_steer = z_steer.to(out.dtype)
  # print(f"z steer : {z_steer.shape} , {z_steer.mean()}")
  # print(f"out before steering : {out.shape}, {out.mean()}")
  # out = batch, psn , model
  out[:, timestep] =  out[:, timestep] + z_steer
  # print(f"out after sterring : {out.shape}, {out.mean()}")
  return out

def train_model(model, prompt, int_layer=6, N=500, lr=0.1):
    cfg = model.config
    model = model.to(device)
    model.eval()
    for p in model.parameters():
        p.requires_grad_(False)

    z_steer = torch.empty((1, cfg.hidden_size), device=device)
    xavier_uniform_(z_steer)
    z_steer.requires_grad_(True)

    optimizer = Adam([z_steer], lr=lr)
    token_ids = tokenizer.encode(prompt, return_tensors='pt').to(device)

    for m in model.modules():
        m._forward_hooks.clear()
        m._forward_pre_hooks.clear()

    hook = partial(add_steering_vector, z_steer=z_steer, timestep=0)
    handle = model.model.layers[int_layer].register_forward_hook(hook)

    try:
        for i in range(N):
            logits = model(token_ids).logits          # b s v
            pred = rearrange(logits[:, :-1], "b s v -> b v s")
            target = token_ids[:, 1:]
            loss = cross_entropy(pred, target)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            if i % 20 == 0:
                print(f"Loss is {loss.item():.4f}")
    finally:
        handle.remove()

    return z_steer.detach()

#   handle.remove()
prompt = dev0_100[0]
train_model(model, prompt)

Loss is 5.0078
Loss is 1.6562
Loss is 0.7773
Loss is 0.3567
Loss is 0.1902
Loss is 0.1235
Loss is 0.0890
Loss is 0.0670


KeyboardInterrupt: 

#

Learnings

- That we need to pass z_steer to gpu before passing it into optimiser, otherwise adam updates the cpu tensor.

- That we need to set requires_grad to false for all model.parameters besides from z_steer otherwise gpu memory usage blows up

## L2 Now make batched loop




Q: How to go from single batch to multi-batch
- curently logits =   1, 32000, 10  -> 8, 32000,10

T1 : How to batch encode
Q: What is the difference between tokenizer.encode and tokenizer.tokenize


In [ ]:
type(tokenizer)

transformers.models.llama.tokenization_llama.LlamaTokenizer

In [ ]:
help(tokenizer)

Help on LlamaTokenizer in module transformers.models.llama.tokenization_llama object:

class LlamaTokenizer(transformers.tokenization_utils_tokenizers.TokenizersBackend)
 |  LlamaTokenizer(
 |      vocab: str | dict | list | None = None,
 |      merges: str | list | None = None,
 |      clean_up_tokenization_spaces=False,
 |      unk_token='<unk>',
 |      bos_token='<s>',
 |      eos_token='</s>',
 |      use_default_system_prompt=False,
 |      legacy=False,
 |      add_prefix_space=None,
 |      **kwargs
 |  )
 |
 |  Construct a Llama tokenizer. Based on byte-level Byte-Pair-Encoding.
 |
 |  This uses notably ByteFallback and no normalization.
 |
 |  ```python
 |  >>> from transformers import LlamaTokenizer
 |
 |  >>> tokenizer = LlamaTokenizer.from_pretrained("hf-internal-testing/llama-tokenizer")
 |  >>> tokenizer.encode("Hello this is a test")
 |  [1, 15043, 445, 338, 263, 1243]
 |  ```
 |
 |  If you want to change the `bos_token` or the `eos_token`, make sure to specify them whe

In [ ]:
# 4 mins to get steering vectors

def add_steering_vector(inst, inp, out, z_steer, timestep):
  z_steer = z_steer.to(out.dtype)
  # print(f"z steer : {z_steer.shape} , {z_steer.mean()}")
  # print(f"out before steering : {out.shape}, {out.mean()}")
  # out = batch, psn , model
  out[:, timestep] =  out[:, timestep] + z_steer
  # print(f"out after sterring : {out.shape}, {out.mean()}")
  return out

def train_model(model, prompts, int_layer=6, N=500, lr=0.1):
    batch_size = len(prompts)

    cfg = model.config
    model = model.to(device)
    model.eval()
    for p in model.parameters():
        p.requires_grad_(False)

    z_steer = torch.empty((batch_size, cfg.hidden_size), device=device)
    xavier_uniform_(z_steer)
    z_steer.requires_grad_(True)

    optimizer = Adam([z_steer], lr=lr)

    #how to do tokenizer encode for different len prompts
    #tokenizer ecnoder converts str -> token representation

    #token_ids : tensor [b, seq_len + pad] where seq_len +pad is max_seq size
    out = tokenizer(prompts, padding = True, padding_side='right', return_tensors="pt").to(device)
    token_ids,  attention_mask = out.input_ids, out.attention_mask

    for m in model.modules():
        m._forward_hooks.clear()
        m._forward_pre_hooks.clear()

    hook = partial(add_steering_vector, z_steer=z_steer, timestep=0)
    handle = model.model.layers[int_layer].register_forward_hook(hook)

    try:
        for i in range(N):
            logits = model(token_ids, attention_mask= attention_mask).logits          # b s v
            pred = rearrange(logits[:, :-1], "b s v -> b v s")
            target = token_ids[:, 1:].clone()
            target[attention_mask[:, 1:] == 0] = -100
            loss = cross_entropy(pred, target)   # -100 is ignored by default

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            if i % 20 == 0:
                print(f"Loss is {loss.item():.4f}")
            if loss.item() < 0.05:
              break
    finally:
        handle.remove()

    return z_steer.detach()

batch_size = 32
batches = [dev1_100[i:i + batch_size] for i in range(0, len(dev1_100), batch_size)]

z_steers = []
for batch in batches:
  z_steer = train_model(model, batch) #expect tensor [ 2, 4097]
  z_steers.append(z_steer)

torch.save(z_steers, "target_tensors.pt")

Loss is 3.9844
Loss is 1.5820
Loss is 0.5811
Loss is 0.2449
Loss is 0.1182
Loss is 0.0651
Loss is 4.0703
Loss is 1.4766
Loss is 0.5332
Loss is 0.2307
Loss is 0.1192
Loss is 0.0710
Loss is 4.4023
Loss is 1.6074
Loss is 0.6196
Loss is 0.3081
Loss is 0.1423
Loss is 0.0648
Loss is 4.4297
Loss is 1.4893
Loss is 0.4370
Loss is 0.1604
Loss is 0.0587


In [ ]:
torch.save(z_steers, "source_tensors.pt")

Q: How to encode a batch of sentences
batch = tokenizer(sentences, padding=True, return_tensors="pt").t(device)

## U1 Understanding padding

E1 :

In [ ]:
tokenizer.eos_token

'</s>'

In [ ]:
tokenizer.pad_token_id

29871

In [ ]:
help(tokenizer.encode)

Help on method encode in module transformers.tokenization_utils_base:

encode(
    text: 'TextInput | PreTokenizedInput | EncodedInput',
    text_pair: 'TextInput | PreTokenizedInput | EncodedInput | None' = None,
    add_special_tokens: 'bool' = True,
    padding: 'bool | str | PaddingStrategy' = False,
    truncation: 'bool | str | TruncationStrategy | None' = None,
    max_length: 'int | None' = None,
    stride: 'int' = 0,
    padding_side: 'str | None' = None,
    return_tensors: 'str | TensorType | None' = None,
    **kwargs
) -> 'list[int]' method of transformers.models.llama.tokenization_llama.LlamaTokenizer instance
    Converts a string to a sequence of ids (integer), using the tokenizer and vocabulary.

    Same as doing `self.convert_tokens_to_ids(self.tokenize(text))`.

    Args:
        text (`str`, `list[str]` or `list[int]`):
            The first sequence to be encoded. This can be a string, a list of strings (tokenized string using the
            `tokenize` method) o

In [ ]:
import torch
from torch.nn.functional import pad

prompts = ["The cat sat", "Once upon a time there was a dragon"]
tokenizer.pad_token = tokenizer.eos_token
PAD = tokenizer.pad_token_id


# ---------------------------------------------------------------
# Exercise 1: implement padding yourself
# ---------------------------------------------------------------
def my_pad(prompts, tokenizer, side="right"):
    """
    Tokenize each prompt with tokenizer.encode (no padding),
    then pad them yourself to the length of the longest one.

    Returns:
        input_ids:      LongTensor (batch, max_len)
        attention_mask: LongTensor (batch, max_len), 1 = real, 0 = pad
    Don't use tokenizer(..., padding=...) inside this function.
    """
    # TODO
    tokens0 = tokenizer.encode(prompts[0], return_tensors='pt')
    print(tokens0.shape)

    tokens1 = tokenizer.encode(prompts[1], return_tensors='pt')
    print(tokens1.shape)

    #pad tokens0 -> len(token1)
    print(tokens0.shape, tokens1.shape)

    max_len = tokens1.shape[1] # 10

    #pad tokens 0 -> max_len
    pad_right = max_len - tokens0.shape[1]
    print(pad_right)

    tokens0 = pad(tokens0, (0, pad_right), "constant", value = PAD)

    # attn_mask
    tokens = torch.cat((tokens0, tokens1), dim=0)
    attn_mask = (tokens != 2).int()
    print(attn_mask)

    return tokens, attn_mask







my_pad(prompts, tokenizer)


# ---------------------------------------------------------------
# Exercise 2: build next-token targets that ignore padding
# ---------------------------------------------------------------
def make_targets(input_ids, attention_mask):
    """
    Returns:
        target: LongTensor (batch, max_len - 1)
                target[:, t] is the token the model should predict
                from position t. Pad positions are set to -100.
    """
    # TODO
    pass

torch.Size([1, 4])
torch.Size([1, 10])
torch.Size([1, 4]) torch.Size([1, 10])
6
tensor([[1, 1, 1, 1, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]], dtype=torch.int32)


In [ ]:
import torch as t
a1 = t.tensor([[1,2,3]]) # 1, 3
a2 = t.tensor([[1,2,3]]) # 1, 3
a3 = t.cat((a1, a2), dim=0)
a3

tensor([[1, 2, 3],
        [1, 2, 3]])

In [ ]:
#attn_mask
a1 = t.tensor([1,2,3, 0, 0])
mask = a1 != 0
mask.int()

tensor([1, 1, 1, 0, 0], dtype=torch.int32)

In [ ]:
# pad this to 3 , 8
from torch.nn.functional import pad
padding = (0 , 3)
x = torch.randn((3, 5))
x = pad(x, padding, mode = "constant" , value=0)
x

tensor([[-1.1170, -0.0295, -0.0422,  1.3622,  0.7684,  0.0000,  0.0000,  0.0000],
        [ 1.1585, -0.4935,  0.3465, -0.6532, -0.9262,  0.0000,  0.0000,  0.0000],
        [-0.8606, -0.4981,  2.0555,  0.0752,  0.9092,  0.0000,  0.0000,  0.0000]])

Q: What is an causal mask
A lower triangular matrix, that masks/ covers, future tokens for autrogressive models

Q: What is a padding mask, Why do we need one?
- EX:
padded_tokens:  [1, 120, 360, 42, 8, 0, 0, 0]
padding mask:   [1,   1,   1,  1, 1, 0, 0, 0]
- So when we pass it into teh model(attention_mask = padding_mask) function which calls self.__call__ . When we calcualte attention in teh attention layers, it attruvtes no weight to these padding dimesnions



In [ ]:
# build attn_mask
x = torch.randn((3, 5))



## 3 Calculate z_score and z_target

In [ ]:
target_tensors = torch.load('target_tensors.pt')
source_tensors = torch.load('source_tensors.pt')

def get_zscore(t_targets, t_sources, lam):
  z_target = torch.cat(t_targets)
  z_target_mean = z_target.mean(dim=0)

  z_source = torch.cat(t_sources)
  z_source_mean = z_source.mean(dim=0)

  ## taking the difference between the negative and positive sentiment vectors
  z_totarget = z_target_mean - z_source_mean
  return z_totarget
  ## the negative sentiment vector + scaled difference vector
  ## motivation: recover the positive sentiment vector
  ## use this as the offset in the hook
  z_target_pred = z_source + lam * z_totarget

z_target_pred = get_zscore(target_tensors, source_tensors, lam= 0.5)



In [ ]:
z_target_pred.shape

torch.Size([4096])

# 4 Intervene on our model with z_totarget instead

In [ ]:
## how2 access the module list -> how 2 access a single LlamaDecoderLayer
from functools import partial

# res stream : batch, psn , dmodel  | 1, 30, 4096

def add_steering_vector(inst, inp, out, z_steer, timestep):
  z_steer = z_steer.to(out.dtype)
  # print(f"z steer : {z_steer.shape} , {z_steer.mean()}")
  # print(f"out before steering : {out.shape}, {out.mean()}")
  # out = batch, psn , model
  out[:, timestep] =  out[:, timestep] + z_steer
  # print(f"out after sterring : {out.shape}, {out.mean()}")
  return out

In [ ]:
import torch as t
from torch.nn.init import xavier_normal_
from torch.nn.functional import cross_entropy
from torch.optim import Adam
import einops
from einops import rearrange

def train_loop(model, token_ids, N, optimizer):       # ADAM: takes optimizer instead of lr
    cross_ent = []
    for i in range(N):
        logits = model(token_ids).logits
        pred = rearrange(logits[:, :-1], "b s v -> b v s")     # FIX: shift, predictions 0..n-2
        target = token_ids[:, 1:]                              # FIX: shift, the next tokens
        loss = cross_entropy(pred, target)
        loss.backward()

        optimizer.step()                                       # ADAM: replaces the manual update
        optimizer.zero_grad()                                  # ADAM: resets z_steer.grad

        cross_ent.append(loss.item())                          # FIX: store number, not tensor
        if i % 20 == 0:
            print(f"step {i}, loss = {loss.item():.4f}")
        if loss.item() < 0.01:                                 # early stop once learned
            break
    return cross_ent

def steer_model(model, reference, z_steer, int_layer=6,):
    """
    Optimize a steering vector so the frozen model reconstructs `reference`.
    z_steer is added to the output of layer `int_layer` at the first timestep.

    returns:
        cross_ent: list of losses
        z_steer: the trained steering vector
    """
    model = model.to(device)
    model.requires_grad_(False)                                # FIX: freeze the model

    cfg = model.config
    z_steer = t.empty(1, cfg.hidden_size, device=device)
    xavier_normal_(z_steer)
    z_steer.requires_grad_(True)

    inputs = tokenizer(reference, return_tensors="pt")         # FIX: no " ".join on a string
    eos = t.tensor([[tokenizer.eos_token_id]])
    token_ids = t.cat([inputs.input_ids, eos], dim=1).to(device)

    N = 500
    lr = 0.1
    optimizer = Adam([z_steer], lr=lr)                         # ADAM: create optimizer

    for m in model.modules():
        m._forward_hooks.clear()
        m._forward_pre_hooks.clear()

    hook = partial(add_steering_vector, z_steer=z_steer, timestep=0)   # FIX: 6 → 0
    handle = model.model.layers[int_layer].register_forward_hook(hook)

    try:
        cross_ent = train_loop(model, token_ids, N, optimizer)
    finally:
        handle.remove()                                        # FIX: always remove the hook

    return model , cross_ent, z_steer.detach()

prompt = dev0_100[49]
print(prompt)
model, cross_ent, z_steer = steer_model(model, prompt, z_steer = z_target_pred)
                                         # should end near 0

it was just terrible !

step 0, loss = 6.2891
step 20, loss = 3.2637
step 40, loss = 1.4189
step 60, loss = 0.5649
step 80, loss = 0.2515
step 100, loss = 0.1202
step 120, loss = 0.0570
step 140, loss = 0.0297
step 160, loss = 0.0177
step 180, loss = 0.0119


In [ ]:
prompt

'it was just terrible !\n'

In [ ]:
tokens = tokenizer(prompt).input_ids

start_token = torch.tensor([[tokens[1]]])

token_pred = model.generate(start_token)

tokenizer.batch_decode(token_pred)

['it is a good idea to have a backup plan.\nThe first thing you should do is to make']

In [ ]:
dev0_100[49]

'it was just terrible !\n'

In [ ]:
help(model)

Help on LlamaForCausalLM in module transformers.models.llama.modeling_llama object:

class LlamaForCausalLM(LlamaPreTrainedModel, transformers.generation.utils.GenerationMixin)
 |  LlamaForCausalLM(config)
 |
 |  The Llama Model for causal language modeling.
 |
 |  This model inherits from [`PreTrainedModel`]. Check the superclass documentation for the generic methods the
 |  library implements for all its model (such as downloading or saving, resizing the input embeddings, pruning heads
 |  etc.)
 |
 |  This model is also a PyTorch [torch.nn.Module](https://pytorch.org/docs/stable/nn.html#torch.nn.Module) subclass.
 |  Use it as a regular PyTorch Module and refer to the PyTorch documentation for all matter related to general usage
 |  and behavior.
 |
 |  Parameters:
 |      config ([`LlamaForCausalLM`]):
 |          Model configuration class with all the parameters of the model. Initializing with a config file does not
 |          load the weights associated with the model, only the 

ImportError: cannot import name 'HookedTransformer' from 'transformers' (/usr/local/lib/python3.13/dist-packages/transformers/__init__.py)

In [ ]:
dir(model)

['T_destination',
 '__annotations__',
 '__call__',
 '__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__firstlineno__',
 '__format__',
 '__ge__',
 '__getattr__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__setstate__',
 '__sizeof__',
 '__static_attributes__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_adjust_bias',
 '_adjust_missing_and_unexpected_keys',
 '_apply',
 '_assisted_decoding',
 '_auto_class',
 '_backward_compatibility_gradient_checkpointing',
 '_backward_hooks',
 '_backward_pre_hooks',
 '_beam_search',
 '_beam_search_has_unfinished_sequences',
 '_buffers',
 '_call_impl',
 '_can_compile_fullgraph',
 '_can_record_outputs',
 '_can_set_attn_implementation',
 '_can_set_experts_implementation',
 '_can_set_experts_implementation_cached_value',
 '_check_and_adjust_attn_

In [ ]:
help(model.generate)

Help on method generate in module transformers.generation.utils:

generate(
    inputs: torch.Tensor | None = None,
    generation_config: transformers.generation.configuration_utils.GenerationConfig | None = None,
    logits_processor: transformers.generation.logits_process.LogitsProcessorList | None = None,
    stopping_criteria: transformers.generation.stopping_criteria.StoppingCriteriaList | None = None,
    prefix_allowed_tokens_fn: collections.abc.Callable[[int, torch.Tensor], list[int]] | None = None,
    synced_gpus: bool | None = None,
    assistant_model: Optional[ForwardRef('PreTrainedModel')] = None,
    streamer: Optional[ForwardRef('BaseStreamer')] = None,
    negative_prompt_ids: torch.Tensor | None = None,
    negative_prompt_attention_mask: torch.Tensor | None = None,
    custom_generate: str | collections.abc.Callable | None = None,
    **kwargs
) -> transformers.generation.utils.GenerateDecoderOnlyOutput | transformers.generation.utils.GenerateEncoderDecoderOutput | 

# 5 Roberta

Q: What type of model is Bert
Encoder

Q: What is bert used for here?
Semantic classification. Measure how well sentiment of sentence is flipped by using bert for sentiment classifcation

Q: How to use bert for semantic classification
https://www.geeksforgeeks.org/nlp/fine-tuning-bert-model-for-sentiment-analysis/




In [ ]:
import pandas as pd
import torch

from torch.utils.data import Dataset, DataLoader
from transformers import (
    BertTokenizerFast,
    BertForSequenceClassification,
    DataCollatorWithPadding
)
from transformers import AutoTokenizer

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report


In [ ]:
!git clone https://github.com/shentianxiao/language-style-transfer.git

fatal: destination path 'language-style-transfer' already exists and is not an empty directory.


In [ ]:
import random
path0 = './language-style-transfer/data/yelp/sentiment.dev.0'
path1 = './language-style-transfer/data/yelp/sentiment.dev.1'

with open(path0) as f:
  # help(f)
  dev0_sentences= f.readlines()
  print(len(dev0_sentences))

with open(path1) as f:
  dev1_sentences= f.readlines()
  print(len(dev1_sentences))



25278
38205


In [ ]:
tokenizer = AutoTokenizer.from_pretrained("google-bert/bert-base-uncased")
model = BertForSequenceClassification.from_pretrained("google-bert/bert-base-uncased")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
import pandas

## add a positive and negative col to the dataset

df = pd.DataFrame({
    "text": dev0_sentences + dev1_sentences,
    "label": [0] * len(dev0_sentences) + [1] * len(dev1_sentences),
})

df.iloc[0]

,0
text,ok never going back to this place again .\n
label,0


In [ ]:

import torch
from torch.utils.data import Dataset, DataLoader

class YelpDataset(Dataset):
    def __init__(self, df):
        self.texts = df["text"].tolist()
        self.labels = df["label"].tolist()

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return self.texts[idx], self.labels[idx]


def collate_fn(batch):
    texts, labels = zip(*batch)              # split tuples into two lists
    enc = tokenizer(
        list(texts),
        padding=True,
        truncation=True,
        return_tensors="pt",
    )
    enc["labels"] = torch.tensor(labels)
    return enc


dataset = YelpDataset(df)
dl_dev = DataLoader(dataset, batch_size=32, shuffle=True, collate_fn=collate_fn,
                    num_workers=4, prefetch_factor=2)



In [ ]:
from torch.optim import Adam

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

def train_bert(model, dl, lr=2e-5, epochs=1):
    model.to(device)
    model.train()
    model.requires_grad_(True)

    optimizer = Adam(model.parameters(), lr=lr)

    for epoch in range(epochs):
        for step, batch in enumerate(dl):
            batch = {k: v.to(device) for k, v in batch.items()}

            output = model(**batch)          # input_ids, attention_mask, labels
            loss = output.loss

            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

            if step % 5 == 0:
                print(f"epoch {epoch} step {step} loss {loss.item():.3f}")
            if step == 100 :
                break
    return loss.item()


train_bert(model, dl_dev)

epoch 0 step 0 loss 0.037
epoch 0 step 5 loss 0.133
epoch 0 step 10 loss 0.126
epoch 0 step 15 loss 0.198
epoch 0 step 20 loss 0.114
epoch 0 step 25 loss 0.171
epoch 0 step 30 loss 0.076
epoch 0 step 35 loss 0.140
epoch 0 step 40 loss 0.155
epoch 0 step 45 loss 0.145
epoch 0 step 50 loss 0.081
epoch 0 step 55 loss 0.058
epoch 0 step 60 loss 0.302
epoch 0 step 65 loss 0.176
epoch 0 step 70 loss 0.043
epoch 0 step 75 loss 0.116
epoch 0 step 80 loss 0.025
epoch 0 step 85 loss 0.028
epoch 0 step 90 loss 0.042
epoch 0 step 95 loss 0.177
epoch 0 step 100 loss 0.140


0.13998223841190338

In [ ]:

def eval_bert(model):
  model = model.to(device)

  with torch.no_grad():
     for idx, batch in enumerate(dl_dev):
        tokens = batch["input_ids"]
        labels = batch["labels"]
        attention_mask = batch["attention_mask"]
        tokens = tokens.to(device)
        labels = labels.to(device)
        logits = model(tokens, labels = labels, attention_mask = attention_mask).logits
        pred = torch.argmax(probs, dim=-1)
        out = labels == pred
        print(out)
        acc = out.sum() / out.shape[0]
        print(acc)
        break

eval_bert(model)

tensor([ True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True, False,  True,  True, False,  True,  True,
         True,  True], device='cuda:0')
tensor(0.9375, device='cuda:0')


In [ ]:
dl_dev = DataLoader(dataset, batch_size = 32, shuffle =True, num_workers= 2, prefetch_factor=1)

batch = next(iter(dl_dev))

tokens = tokenizer(batch, padding = True, padding_side='right', return_tensors="pt").input_ids
tokens = tokens.to(device)
model = model.to(device)
model(tokens).logits.shape



In [ ]:
help(torch.utils.data.DataLoader)

In [ ]:
 help(BertForSequenceClassification)

In [ ]:
help(tokenizer)

In [ ]:
help(BertForSequenceClassification.from_pretrained)